# 🛍️ Customer Shopping Behavior - End-to-End Data Analytics Portfolio Project
### Tech Stack: Python (ETL & EDA) | SQL (Database & Business Intelligence) | Power BI & Interactive Dashboard
---
**Dataset:** Retail Customer Shopping Trends (3,900 rows, 18 features)
**Author:** Data Analytics Portfolio Implementation
**Reference Tutorial:** Amlan Mohanty - *COMPLETE Data Analytics Portfolio Project in 6 EASY Steps*

## 📌 Project Objectives
1. **Exploratory Data Analysis (Python):** Clean, impute, transform, and engineer features on customer shopping transactions.
2. **Database Integration (SQL):** Load cleaned data into a relational database (SQLite / PostgreSQL / MySQL) and run 10 business queries.
3. **Dashboard & Visualization:** Build an interactive KPI dashboard to empower executives with real-time customer intelligence.

## Step 1: Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Load raw dataset
df = pd.read_csv('customer_shopping_behavior.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Step 2: Exploratory Data Analysis (EDA)
Check column types, non-null counts, missing values, and descriptive statistics.

In [ ]:
# Dataframe summary info
df.info()

In [ ]:
# Summary statistics for numerical and categorical variables
df.describe(include='all')

In [ ]:
# Checking missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])

## Step 3: Data Cleaning & Missing Value Imputation
The `Review Rating` column contains 37 missing values. We impute them using the **median review rating for each product Category** to preserve category-level rating distributions.

In [ ]:
# Impute Review Rating by Category median
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
print("Remaining missing values:", df['Review Rating'].isnull().sum())

## Step 4: Standardize Column Names & Feature Engineering
- Convert column names to `snake_case`
- Create `age_group` binning into 4 cohorts: Young Adult, Adult, Middle-aged, Senior
- Map `frequency_of_purchases` to days (`purchase_frequency_days`)
- Create customer segmentation (`customer_segment`: New, Returning, Loyal)
- Verify and drop redundant `promo_code_used` column (identical to `discount_applied`)

In [ ]:
# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

# 1. Age Group Quartiles
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels).astype(str)

# 2. Purchase Frequency in Days
frequency_mapping = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Every 3 Months': 90,
    'Quarterly': 90,
    'Annually': 365
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# 3. Customer Lifecycle Segmentation
def segment_customer(p):
    if p == 1:
        return 'New'
    elif 2 <= p <= 10:
        return 'Returning'
    else:
        return 'Loyal'

df['customer_segment'] = df['previous_purchases'].apply(segment_customer)

# 4. Redundancy check
is_identical = (df['discount_applied'] == df['promo_code_used']).all()
print(f"Are discount_applied and promo_code_used identical? {is_identical}")
if is_identical:
    df = df.drop(columns=['promo_code_used'])

print("Cleaned columns:", df.columns.tolist())
df.head()

## Step 5: Save Cleaned Data and Load into SQLite Database

In [ ]:
# Save cleaned dataset
df.to_csv('cleaned_customer_shopping_behavior.csv', index=False)

# Connect to SQLite and create customer table
conn = sqlite3.connect('customer_behavior.db')
df.to_sql('customer', conn, if_exists='replace', index=False)
print("Loaded cleaned data into table 'customer' in 'customer_behavior.db'.")

## Step 6: SQL Business Analysis (10 Core Queries)
We execute the 10 business SQL questions highlighted in the tutorial directly on the database.

In [ ]:
# Helper function to execute and display query results
def run_query(title, sql):
    print(f"=== {title} ===")
    res = pd.read_sql(sql, conn)
    display(res)
    return res

In [ ]:
# Q1: Revenue generated by male vs. female customers
q1 = """SELECT gender, SUM(purchase_amount) AS revenue, COUNT(customer_id) AS total_orders,
ROUND(AVG(purchase_amount), 2) AS avg_order_value
FROM customer GROUP BY gender ORDER BY revenue DESC;"""
run_query("Q1: Revenue by Gender", q1)

In [ ]:
# Q2: Customers who used a discount but spent more than the average purchase amount
q2 = """SELECT customer_id, gender, category, item_purchased, purchase_amount
FROM customer
WHERE discount_applied = 'Yes' AND purchase_amount >= (SELECT AVG(purchase_amount) FROM customer)
ORDER BY purchase_amount DESC LIMIT 10;"""
run_query("Q2: High Spenders with Discount", q2)

In [ ]:
# Q3: Top 5 products with highest average review rating
q3 = """SELECT item_purchased, ROUND(AVG(review_rating), 2) AS avg_rating, COUNT(customer_id) AS review_count
FROM customer GROUP BY item_purchased ORDER BY avg_rating DESC LIMIT 5;"""
run_query("Q3: Top 5 Products by Rating", q3)

In [ ]:
# Q4: Compare average purchase amount between Standard and Express Shipping
q4 = """SELECT shipping_type, COUNT(customer_id) AS total_orders, ROUND(AVG(purchase_amount), 2) AS avg_spend,
ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM customer WHERE shipping_type IN ('Standard', 'Express') GROUP BY shipping_type;"""
run_query("Q4: Standard vs Express Shipping Spend", q4)

In [ ]:
# Q5: Spend and total revenue comparison between subscribers and non-subscribers
q5 = """SELECT subscription_status, COUNT(customer_id) AS total_customers,
ROUND(100.0 * COUNT(customer_id) / (SELECT COUNT(*) FROM customer), 2) AS customer_pct,
ROUND(AVG(purchase_amount), 2) AS avg_spend, ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM customer GROUP BY subscription_status ORDER BY total_revenue DESC;"""
run_query("Q5: Subscribers vs Non-subscribers", q5)

In [ ]:
# Q6: Top 5 products with highest percentage of purchases with discounts
q6 = """SELECT item_purchased, COUNT(*) AS total_orders,
SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) AS discount_orders,
ROUND(100.0 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS discount_rate_pct
FROM customer GROUP BY item_purchased ORDER BY discount_rate_pct DESC LIMIT 5;"""
run_query("Q6: Top 5 Discounted Products", q6)

In [ ]:
# Q7: Customer segmentation count into New, Returning, and Loyal
q7 = """WITH customer_type AS (
    SELECT customer_id, previous_purchases,
    CASE WHEN previous_purchases = 1 THEN 'New'
         WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
         ELSE 'Loyal' END AS customer_segment
    FROM customer
)
SELECT customer_segment, COUNT(*) AS number_of_customers,
ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer), 2) AS pct_share
FROM customer_type GROUP BY customer_segment ORDER BY number_of_customers DESC;"""
run_query("Q7: Customer Segmentation", q7)

In [ ]:
# Q8: Top 3 most purchased products within each category (Window Functions)
q8 = """WITH item_counts AS (
    SELECT category, item_purchased, COUNT(customer_id) AS total_orders,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM customer GROUP BY category, item_purchased
)
SELECT item_rank, category, item_purchased, total_orders
FROM item_counts WHERE item_rank <= 3 ORDER BY category, item_rank;"""
run_query("Q8: Top 3 Products per Category", q8)

In [ ]:
# Q9: Are repeat buyers (>5 purchases) likely to subscribe?
q9 = """SELECT subscription_status, COUNT(customer_id) AS repeat_buyers,
ROUND(100.0 * COUNT(customer_id) / (SELECT COUNT(*) FROM customer WHERE previous_purchases > 5), 2) AS pct_share
FROM customer WHERE previous_purchases > 5 GROUP BY subscription_status;"""
run_query("Q9: Repeat Buyers Subscription Status", q9)

In [ ]:
# Q10: Revenue contribution of each age group
q10 = """SELECT age_group, COUNT(customer_id) AS total_customers, SUM(purchase_amount) AS total_revenue,
ROUND(100.0 * SUM(purchase_amount) / (SELECT SUM(purchase_amount) FROM customer), 2) AS revenue_share_pct
FROM customer GROUP BY age_group ORDER BY total_revenue DESC;"""
run_query("Q10: Revenue by Age Group", q10)

## Step 7: Summary & Interactive Dashboard Launch
To launch the interactive dashboard on `http://localhost:8501`, double click `run_dashboard.bat` or run:
```bash
streamlit run dashboard.py
```